# 02 · Construct tokens and stochastic masks


In [ ]:
from pathlib import Path
import json, os, sys

# Find the checkout/release from the notebook's working directory.
ROOT = Path(os.environ.get('GF_ROOT', Path.cwd())).resolve()
while not (ROOT / 'src/gavd6_sjepa').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'src/gavd6_sjepa').is_dir(), 'Open this notebook from the GAVD6 checkout or release.'
sys.path.insert(0, str(ROOT / 'notebooks/gait_fidelity'))
sys.path.insert(0, str(ROOT / 'src'))
from tutorial_helpers import configure, preview_images
study = configure(ROOT)


## Define a token before defining a mask

For the body-12 model, one token contains **four adjacent frames of one joint**.
For a window of $T$ frames, patch size $p=4$, and $J=12$ joints, there are
$S=T/p$ patches and $N=SJ$ tokens. Token $(s,j)$ is flattened at index $sJ+j$.
Source windows have 128 frames, hence 384 tokens; the smaller software fixture
uses the dimensions saved in its configuration.

A naturally unobserved joint is different from an artificially hidden query.
We may add an artificial mask only when a patch contains at least one observed
frame. Naturally missing frames also create pretraining queries, as notebook
04 shows. The
model still retains output locations for all patches and joints, including
those with no observations. A reference-valid mask never determines which
input locations are eligible for masking.

For the worked example, we prefer a correctly named, occluded training
baseline with at least two observed tokens, using manifest order to break ties.
If none qualifies, we use the first training row with that input support. We
report sparse candidates passed over for the picture and keep them in the
study; explicit sparse-grid tests below still verify their sampler behavior.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from gavd6_sjepa.research_directions.gait_fidelity.data import load_dataset
from gavd6_sjepa.research_directions.gait_fidelity.masking import (
    sample_mask, patch_support, POLICIES, REGIONS, EDGES, coverage_audit, audit_matching)
from gavd6_sjepa.research_directions.synthetic_training_v2.contracts import JOINTS

bundle = load_dataset(study.bundle_path())
records = pd.DataFrame(bundle.records)
config = study.artifact('config.json')
# This metadata rule deliberately includes natural missingness when available.
chosen = records.index[records['split'].eq('train') & records['movement_state'].eq('baseline')
                       & records['naming'].eq('correct') & records['observation'].eq('occluded')]
p = int(config['model']['patch_size'])
candidates = list(dict.fromkeys([*chosen, *records.index[records['split'].eq('train')]]))
row_id, sparse_candidates = None, 0
for candidate in candidates:
    candidate_observed = bundle.inputs['observed'][candidate]
    candidate_tokens = candidate_observed.reshape(-1, p, 12).any(axis=1)
    if candidate_tokens.sum() >= 2:
        row_id = int(candidate)
        break
    sparse_candidates += 1
assert row_id is not None, 'No training row has two observed tokens; inspect extraction before continuing.'
print('Sparse candidates passed over for this demonstration:', sparse_candidates,
      '(all remain in the study)')
raw = {key: value[[row_id]].copy() for key, value in bundle.inputs.items()}
observed = raw['observed']
B, T, J = observed.shape
S = T // p
assert T % p == 0 and J == 12
eligible = observed.reshape(B, S, p, J).any(axis=2)
np.testing.assert_array_equal(eligible, patch_support(observed, p))
fraction = float(config['training']['mask_fraction'])
available_count = eligible.sum(axis=(1, 2))
budgets = np.array([min(max(1, round(int(n) * fraction)), int(n) - 1) if n > 1 else 0
                    for n in available_count])
display(pd.DataFrame({'frames': [T], 'patches': [S], 'joints': [J], 'grid tokens': [S * J],
                      'eligible tokens': available_count, 'hidden budget': budgets}))
print('Natural missing joint-frame values:', int((~observed).sum()))


## What the anatomical graph controls

The implementation samples from the connected regions listed below. Each
region is a limb, part of a limb, or a connected torso region in the body-12
graph. It does not grow an unrestricted random subgraph and does not weight
clinically interesting joints more heavily. Overlapping region definitions
can still give some joints a higher selection probability, which we audit.

For each sampled region, draw a duration from 1 through $S/2$ patches and a
uniform start patch. Time wraps cyclically at the window boundary to avoid
automatically under-sampling the edges. The final region can be truncated
when the exact token budget is reached. A wrapped interval appears as two
separate pieces in the model's finite window.


In [ ]:
neighbors = {joint: set() for joint in range(J)}
for a, b in EDGES:
    neighbors[a].add(b)
    neighbors[b].add(a)

def connected(region):
    reached, pending = set(), [region[0]]
    while pending:
        joint = pending.pop()
        if joint in reached:
            continue
        reached.add(joint)
        pending.extend(neighbors[joint].intersection(region) - reached)
    return reached == set(region)

assert all(connected(region) for region in REGIONS)
display(pd.DataFrame([{'region': k, 'joints': ', '.join(JOINTS[j] for j in region),
                       'connected': connected(region)} for k, region in enumerate(REGIONS)]))


## Implement the masking policies in ordinary NumPy

The sampler below shows the complete decisions applied to a single example.
Uniform-token masking samples eligible positions without replacement. Time
blocks visit consecutive patches, using a shuffled joint order if the final
patch only partly fits the budget. Graph-time masking places anatomical
regions over cyclic intervals. Its two controls retain the region-size rule:
one relabels the graph by a fixed permutation, while the other draws random
joints for every interval.

Every policy hides the same number of eligible tokens and leaves at least one
context token when two or more exist. A bounded retry loop plus a declared
uniform fill handles sparse observations. The code uses its own NumPy
generator, so these examples cannot advance a training job's random state.


In [ ]:
def explicit_mask(available, policy, *, seed, fraction=.5, topology_seed=271828):
    rng = np.random.default_rng(seed)
    available = np.asarray(available, dtype=bool)
    blocks, joints = available.shape
    n = int(available.sum())
    budget = min(max(1, round(n * fraction)), n - 1) if n > 1 else 0
    hidden = np.zeros_like(available)
    trace = []
    if budget == 0:
        return hidden, trace
    if policy == 'uniform_tokens':
        hidden.flat[rng.choice(np.flatnonzero(available), budget, replace=False)] = True
    elif policy == 'time_blocks':
        joint_order = rng.permutation(joints)
        start = int(rng.integers(blocks))
        order = [(int((start + k) % blocks), int(j))
                 for k in range(blocks) for j in joint_order if available[(start + k) % blocks, j]]
        for time_patch, joint in order[:budget]:
            hidden[time_patch, joint] = True
    else:
        assert policy in ('graph_time', 'shuffled_topology', 'random_joint_intervals')
        topology = np.random.default_rng(topology_seed).permutation(joints)
        maximum_duration, attempts = max(1, blocks // 2), 0
        while hidden.sum() < budget and attempts < 100 * blocks:
            attempts += 1
            region = np.asarray(REGIONS[int(rng.integers(len(REGIONS)))])
            duration = int(rng.integers(1, maximum_duration + 1))
            start = int(rng.integers(blocks))
            if policy == 'shuffled_topology':
                region = topology[region]
            elif policy == 'random_joint_intervals':
                region = rng.choice(joints, len(region), replace=False)
            before = int(hidden.sum())
            for offset in range(duration):
                for joint in rng.permutation(region):
                    time_patch = (start + offset) % blocks
                    if available[time_patch, joint] and not hidden[time_patch, joint]:
                        hidden[time_patch, joint] = True
                        if hidden.sum() == budget:
                            break
                if hidden.sum() == budget:
                    break
            trace.append({'region': tuple(int(j) for j in region), 'start patch': start,
                          'requested duration': duration, 'new hidden tokens': int(hidden.sum()) - before})
        if hidden.sum() < budget:
            remaining = np.flatnonzero(available & ~hidden)
            hidden.flat[rng.choice(remaining, budget - int(hidden.sum()), replace=False)] = True
    assert int(hidden.sum()) == budget
    assert not (hidden & ~available).any()
    assert n <= 1 or (available & ~hidden).any()
    return hidden, trace

manual, graph_trace = explicit_mask(eligible[0], 'graph_time', seed=17, fraction=fraction)
display(pd.DataFrame(graph_trace).head(12))
print('Sampling trace above shows interval requests; overlapping requests can add fewer tokens.')


### Verify exact agreement, including sparse inputs

Matching only the hidden-token count would miss a joint-order or random-draw
error. The following checks compare every mask bit against the production
sampler for all five policies, several seeds, the selected observations, an
entirely observed grid, an empty grid, and a grid with one eligible token.
The empty and one-token cases intentionally add no artificial mask.


In [ ]:
one_token = np.zeros_like(observed)
one_token[:, 0, 0] = True
test_observations = [observed, np.ones_like(observed), np.zeros_like(observed), one_token]
checked = 0
for test_observed in test_observations:
    test_eligible = test_observed.reshape(B, S, p, J).any(axis=2)
    for policy in POLICIES:
        for seed in (0, 17, 29):
            inline, _ = explicit_mask(test_eligible[0], policy, seed=seed, fraction=fraction)
            actual, receipt = sample_mask(test_observed, policy, rng=np.random.default_rng(seed),
                fraction=fraction, patch_size=p, return_receipt=True)
            np.testing.assert_array_equal(inline, actual[0])
            assert int(inline.sum()) == receipt['hidden_tokens'][0]
            checked += 1
print(f'{checked} complete mask arrays match production bit for bit.')


## Construct the five channels and pack the tokens

For each frame and joint, the encoder receives
$[\widetilde{x},\widetilde{y},c,a,\tau]$: normalized position, native confidence,
context availability, and seconds since the window starts. Artificially
hidden positions and scores become zero. Naturally missing coordinates also
become zero, and availability distinguishes these placeholders from valid
coordinates at zero. Native finite scores retain production's value even
when a location is naturally unobserved; missing scores are adapted to zero.

Grouping four frames produces a vector of $4\times5=20$ numbers for each
patch/joint. A learned linear projection maps this vector into a feature
embedding in notebook 03. Time and joint positions are then added, allowing
the model to distinguish identical numerical values at different locations.


In [ ]:
from gavd6_sjepa.research_directions.gait_fidelity.training import normalize_batch

hidden, _ = sample_mask(observed, 'graph_time', rng=np.random.default_rng(17),
                       fraction=fraction, patch_size=p, return_receipt=True)
normalized, origin, scale, _ = normalize_batch(raw, hidden, patch_size=p)
artificial = np.repeat(hidden, p, axis=1)
usable = observed & ~artificial
safe_xy = np.where(usable[..., None], normalized['xy'], 0.)
safe_confidence = np.where(~artificial, normalized['confidence'], 0.)
seconds = normalized['timestamps'] - normalized['timestamps'][:, :1]
clock = np.broadcast_to(seconds[:, :, None, None], (B, T, J, 1))
channels = np.concatenate([safe_xy, safe_confidence[..., None],
                           usable[..., None].astype(np.float32), clock], axis=-1)
patches = channels.reshape(B, S, p, J, 5).transpose(0, 1, 3, 2, 4).reshape(B, S * J, p * 5)
context_support = usable.reshape(B, S, p, J).any(axis=2).reshape(B, S * J)
assert channels.shape == (B, T, J, 5)
assert patches.shape == (B, S * J, p * 5)
assert np.isfinite(patches).all()
assert not context_support.reshape(B, S, J)[hidden].any()

# Check the indexing with the corresponding unflattened slice.
supported_slots = np.flatnonzero(context_support[0])
assert len(supported_slots), 'This demonstration requires at least one context token.'
token_index = int(supported_slots[0])
time_patch, joint = divmod(token_index, J)
expected = channels[0, time_patch * p:(time_patch + 1) * p, joint].reshape(-1)
np.testing.assert_array_equal(patches[0, token_index], expected)
display(pd.DataFrame(patches[0, token_index].reshape(p, 5),
                     columns=['normalized x', 'normalized y', 'native score', 'context available', 'seconds']))
print({'token_index': token_index, 'patch': time_patch, 'joint': JOINTS[joint],
       'packed_shape': patches.shape, 'context_tokens': int(context_support.sum())})


## Compare the visible masks and their coverage

Orange cells are artificially hidden queries, blue cells retain observed
context, and gray cells have no observed frame in the patch. The same input
and seed are used for all policies, so the exact hiding counts are directly
comparable. The patterns need not have equal duration or per-joint frequency.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from io import BytesIO
from IPython.display import Image

fig, axes = plt.subplots(len(POLICIES), 1, figsize=(12, 13.5), constrained_layout=True)
colormap = ListedColormap(['#dfdfdf', '#c9e4f6', '#e89045'])
mask_rows = []
for ax, policy in zip(axes, POLICIES):
    policy_mask, receipt = sample_mask(observed, policy, rng=np.random.default_rng(17),
        fraction=fraction, patch_size=p, return_receipt=True)
    state = eligible[0].astype(int)
    state[policy_mask[0]] = 2
    ax.imshow(state.T, aspect='auto', interpolation='nearest', cmap=colormap, vmin=0, vmax=2)
    ax.set_yticks(range(J), JOINTS, fontsize=8)
    ax.set(title=policy.replace('_', ' '), xlabel=f'Temporal patch ({p} frames each)')
    mask_rows.append({'policy': policy, 'hidden tokens': receipt['hidden_tokens'][0],
                      'context tokens': receipt['context_tokens'][0],
                      'sparse-fill tokens': receipt['sparse_uniform_fill_tokens']})
axes[0].legend(handles=[Patch(color=color, label=label) for color, label in
                      zip(colormap.colors, ['Naturally unavailable', 'Visible context', 'Hidden query'])],
               loc='upper center', bbox_to_anchor=(.5, 1.47), ncol=3, fontsize=9)
fig.suptitle(('Software fixture' if study.fixture else 'Selected training example') +
             ' — equal budgets, different query patterns', fontsize=13)
png = BytesIO()
fig.savefig(png, format='png', dpi=120, bbox_inches='tight')
display(Image(data=png.getvalue()))
plt.close(fig)
display(pd.DataFrame(mask_rows))


For a bank of $K$ fresh masks, the empirical hiding probability at patch/joint
$(s,j)$ is the number of eligible examples hidden there divided by the number
of eligible examples drawn. Counting natural absence as hiding would corrupt
this denominator. Coverage is a property of repeated draws: a joint can be
hidden throughout one window while still serving as context in other draws.


In [ ]:
draws, demo_seed = 128, 17
mask_rng = np.random.default_rng(demo_seed)
bank = np.stack([sample_mask(observed, 'graph_time', rng=mask_rng, fraction=fraction, patch_size=p)
                 for _ in range(draws)])
denominator = eligible.sum(axis=0) * draws
hidden_count = bank.sum(axis=(0, 1))
probability = np.divide(hidden_count, denominator,
    out=np.full_like(hidden_count, np.nan, dtype=float), where=denominator > 0)
audit = coverage_audit(observed, draws=draws, seed=demo_seed, policy='graph_time',
                       fraction=fraction, patch_size=p)
np.testing.assert_allclose(probability, audit['masked_probability_by_patch_joint'], equal_nan=True)
joint_denominator = denominator.sum(axis=0)
joint_rate = np.divide(hidden_count.sum(axis=0), joint_denominator,
                      out=np.full(J, np.nan), where=joint_denominator > 0)
display(pd.DataFrame({'joint': JOINTS, 'eligible-token hiding probability': joint_rate,
                     'draws with context somewhere': audit['context_somewhere_by_joint']}))
print('Always-hidden eligible slots:', audit['always_hidden_eligible_slots'])
print('Never-hidden eligible slots:', audit['never_hidden_eligible_slots'])

matching = audit_matching(observed, draws=128, seed=17, fraction=fraction, patch_size=p)
display(pd.DataFrame(matching['comparisons']).T)
print(matching['claim_limit'])


The displayed audit is a teaching check on one metadata-selected training
example. The full prepared-run audit retains its own population and settings.
Equal token budgets alone do not isolate anatomical connectivity: graph
relabeling can change joint exposure, and overlapping intervals can change
realized run lengths. Failed matching tolerances remain part of the report
and limit the interpretation of a winning mask; they do not justify removing
an inconvenient control.

The stochastic query mask is used during pretraining. Frozen-readout training
and final restoration receive ordinary observed inputs, without an extra
artificial query mask. Continue to **03_experiment_matrix** for the explicit
encoder, feature predictor, coordinate readout, and registered comparisons.
